# Historical constraint index builder

This notebook rebuilds the historical constraint index from scratch.

The goal is to create a reusable class that:

1. loads `merged_constraints_only.csv`
2. removes noisy item-level constraint rows where `s` starts with `wd:Q` or `wds:Q`
3. normalizes dates
4. exports a raw JSON structure for a selected constraint type

The raw JSON keeps Wikidata predicate IDs, such as `pq:P2305` and `pq:P2303`, rather than replacing them with human-readable names.

In [14]:
import json
import re
import pandas as pd

## Load historical constraints file

In [15]:
df_constraints = pd.read_csv(
    "merged_constraints_only.csv",
    dtype=str
)

df_constraints.head()

,s,p,o,cdate,cuser,ddate,duser
0,wds:Q11379-10e45eb7-44c3-366f-51c7-c33f0b28ba32,pq:P2305,wd:Q130964,2018-10-10T20:01:07Z,Astirmays,2018-12-13T10:23:06Z,Jura1
1,wds:Q11379-10e45eb7-44c3-366f-51c7-c33f0b28ba32,ps:P2302,wd:Q21514353,2018-10-10T19:55:29Z,Astirmays,2018-12-13T10:23:06Z,Jura1
2,wds:Q11379-10e45eb7-44c3-366f-51c7-c33f0b28ba32,pq:P2305,wd:Q25269,2018-10-10T20:01:07Z,Astirmays,2018-12-13T10:23:06Z,Jura1
3,wds:Q11379-10e45eb7-44c3-366f-51c7-c33f0b28ba32,wikibase:rank,wikibase:NormalRank,2018-10-10T19:55:29Z,Astirmays,2018-12-13T10:23:06Z,Jura1
4,wds:Q11379-10e45eb7-44c3-366f-51c7-c33f0b28ba32,pq:P2305,wd:Q296936,2018-10-10T20:01:07Z,Astirmays,2018-12-13T10:23:06Z,Jura1


## Remove noisy item-level constraint rows

Property constraints should be attached to properties, such as `wd:P...` and `wds:P...`.

Some historical rows contain constraints attached to items, such as:

- `wd:Q...`
- `wds:Q...`

These are treated as noise for this analysis and removed.

In [16]:
def remove_item_level_constraint_noise(df):
    mask_noise = (
        df["s"].str.match(r"^wd:Q\d+$", case=False, na=False)
        | df["s"].str.match(r"^wds:Q\d+-", case=False, na=False)
    )

    return df.loc[~mask_noise].copy(), df.loc[mask_noise].copy()


df_constraints_clean, df_removed_q_constraints = remove_item_level_constraint_noise(
    df_constraints
)

{
    "original_rows": len(df_constraints),
    "clean_rows": len(df_constraints_clean),
    "removed_q_rows": len(df_removed_q_constraints),
    "distinct_removed_q_subjects": df_removed_q_constraints["s"].str.lower().nunique(),
}

{'original_rows': 465905,
 'clean_rows': 464409,
 'removed_q_rows': 1496,
 'distinct_removed_q_subjects': 538}

## Normalize dates

We convert `cdate` and `ddate` to UTC-aware timestamps.

In [17]:
for col in ["cdate", "ddate"]:
    df_constraints_clean[col] = pd.to_datetime(
        df_constraints_clean[col],
        errors="coerce",
        utc=True
    )

df_constraints_clean.dtypes

s                     object
p                     object
o                     object
cdate    datetime64[ns, UTC]
cuser                 object
ddate    datetime64[ns, UTC]
duser                 object
dtype: object

## Define constraint type metadata

We begin with one-of and none-of constraints, but the class will accept any constraint type QID.

In [18]:
CONSTRAINT_TYPES = {
    "one_of": "wd:Q21510859",
    "none_of": "wd:Q52558054",
    "symmetric": "wd:Q21510862",
    "inverse": "wd:Q21510855",
    "item_requires_statement": "wd:Q21503247",
    "value_requires_statement": "wd:Q21510864",
    "conflicts_with": "wd:Q21502838",
}

## Class: HistoricalConstraintIndexBuilder

This class stores the cleaned historical constraint dataframe and exports raw constraint dictionaries.

The raw export keeps all `pq:*` qualifier predicates exactly as Wikidata represents them.

For each selected constraint type, the export structure is:

In [19]:
class HistoricalConstraintIndexBuilder:
    def __init__(self, df_constraints):
        self.df_constraints = df_constraints.copy()

    @staticmethod
    def _timestamp_to_json_value(value):
        """
        Convert pandas Timestamp/NaT to JSON-compatible string.

        For raw exports, we keep the readable pandas-style string.
        Later optimized exports can convert open-ended dates to null.
        """
        if pd.isna(value):
            return "NaT"
        return str(value)

    @staticmethod
    def _records_with_dates(df, date_cols):
        """
        Convert a dataframe to records, making timestamps JSON serializable.
        """
        temp = df.copy()

        for col in date_cols:
            if col in temp.columns:
                temp[col] = temp[col].apply(
                    HistoricalConstraintIndexBuilder._timestamp_to_json_value
                )

        return temp.to_dict("records")

    def get_constraint_type_rows(self, constraint_type_qid, constraint_kind):
        """
        Rows where a wds: statement declares a specific constraint type:

            s = wds:P...
            p = ps:P2302
            o = constraint_type_qid
        """
        result = self.df_constraints.loc[
            self.df_constraints["p"].eq("ps:P2302")
            & self.df_constraints["o"].eq(constraint_type_qid),
            ["s", "p", "o", "cdate", "cuser", "ddate", "duser"]
        ].copy()

        result = result.rename(
            columns={
                "s": "constraint_statement",
                "o": "constraint_type_qid",
                "cdate": "constraint_cdate",
                "ddate": "constraint_ddate",
            }
        )

        result["constraint_kind"] = constraint_kind

        return result

    def get_property_links(self, constraint_statement_ids):
        """
        Rows connecting a property to a constraint statement:

            s = wd:P...
            p = p:P2302
            o = wds:P...
        """
        result = self.df_constraints.loc[
            self.df_constraints["p"].eq("p:P2302")
            & self.df_constraints["o"].isin(constraint_statement_ids),
            ["s", "p", "o", "cdate", "ddate"]
        ].copy()

        result = result.rename(
            columns={
                "s": "constrained_property",
                "o": "constraint_statement",
                "cdate": "property_link_cdate",
                "ddate": "property_link_ddate",
            }
        )

        return result

    def get_ranks(self, constraint_statement_ids):
        """
        Rank rows for selected constraint statements:

            s = wds:P...
            p = wikibase:rank
        """
        result = self.df_constraints.loc[
            self.df_constraints["s"].isin(constraint_statement_ids)
            & self.df_constraints["p"].eq("wikibase:rank"),
            ["s", "o", "cdate", "ddate"]
        ].copy()

        result = result.rename(
            columns={
                "s": "constraint_statement",
                "o": "rank",
                "cdate": "rank_cdate",
                "ddate": "rank_ddate",
            }
        )

        return result

    def get_qualifiers(self, constraint_statement_ids):
        """
        Collect all qualifier rows for selected constraint statements.

        These are rows where:

            s = wds:P...
            p starts with pq:

        The output keeps the original pq:PID values.
        """
        result = self.df_constraints.loc[
            self.df_constraints["s"].isin(constraint_statement_ids)
            & self.df_constraints["p"].str.startswith("pq:", na=False),
            ["s", "p", "o", "cdate", "ddate"]
        ].copy()

        result = result.rename(
            columns={
                "s": "constraint_statement",
                "p": "qualifier_pid",
                "o": "entity",
                "cdate": "value_cdate",
                "ddate": "value_ddate",
            }
        )

        return result

    def build_raw_constraint_dict(self, constraint_kind, constraint_type_qid):
        """
        Build raw JSON-ready dictionary for a selected constraint type.

        The export keeps all qualifiers grouped by their Wikidata predicate IDs.
        """
        constraint_type_rows = self.get_constraint_type_rows(
            constraint_type_qid=constraint_type_qid,
            constraint_kind=constraint_kind
        )

        constraint_statement_ids = set(
            constraint_type_rows["constraint_statement"]
        )

        property_links = self.get_property_links(constraint_statement_ids)
        ranks = self.get_ranks(constraint_statement_ids)
        qualifiers = self.get_qualifiers(constraint_statement_ids)

        type_intervals_by_statement = {
            stmt: self._records_with_dates(
                group[
                    [
                        "constraint_type_qid",
                        "constraint_kind",
                        "constraint_cdate",
                        "constraint_ddate",
                    ]
                ],
                ["constraint_cdate", "constraint_ddate"]
            )
            for stmt, group in constraint_type_rows.groupby("constraint_statement")
        }

        property_links_by_statement = {
            stmt: self._records_with_dates(
                group[
                    [
                        "constrained_property",
                        "property_link_cdate",
                        "property_link_ddate",
                    ]
                ],
                ["property_link_cdate", "property_link_ddate"]
            )
            for stmt, group in property_links.groupby("constraint_statement")
        }

        ranks_by_statement = {
            stmt: self._records_with_dates(
                group[
                    [
                        "rank",
                        "rank_cdate",
                        "rank_ddate",
                    ]
                ],
                ["rank_cdate", "rank_ddate"]
            )
            for stmt, group in ranks.groupby("constraint_statement")
        }

        qualifiers_by_statement_and_pid = {}

        for (stmt, qualifier_pid), group in qualifiers.groupby(
            ["constraint_statement", "qualifier_pid"]
        ):
            qualifiers_by_statement_and_pid.setdefault(stmt, {})[qualifier_pid] = (
                self._records_with_dates(
                    group[
                        [
                            "entity",
                            "value_cdate",
                            "value_ddate",
                        ]
                    ],
                    ["value_cdate", "value_ddate"]
                )
            )

        raw_dict = {}

        for constraint_statement, type_intervals in type_intervals_by_statement.items():
            if constraint_statement not in property_links_by_statement:
                continue

            links = property_links_by_statement[constraint_statement]

            # Group property-link intervals by constrained property.
            # Usually there is only one constrained property per wds: statement,
            # but the same statement can have multiple historical p:P2302 intervals.
            links_by_property = {}

            for link in links:
                constrained_property = link["constrained_property"]

                if pd.isna(constrained_property):
                    continue

                links_by_property.setdefault(constrained_property, []).append({
                    "property_link_cdate": link["property_link_cdate"],
                    "property_link_ddate": link["property_link_ddate"],
                })

            for constrained_property, property_link_intervals in links_by_property.items():

                if constrained_property not in raw_dict:
                    raw_dict[constrained_property] = {
                        "constraint_kind": constraint_kind,
                        "statements": {}
                    }

                statement_data = {
                    "constraint_type_qid": constraint_type_qid,
                    "constraint_kind": constraint_kind,
                    "constraint_type_intervals": type_intervals,
                    "property_link_intervals": property_link_intervals,
                    "wikibase:rank": ranks_by_statement.get(constraint_statement, []),
                }

                statement_qualifiers = qualifiers_by_statement_and_pid.get(
                    constraint_statement,
                    {}
                )

                for qualifier_pid, qualifier_records in statement_qualifiers.items():
                    statement_data[qualifier_pid] = qualifier_records

                raw_dict[constrained_property]["statements"][constraint_statement] = (
                    statement_data
                )

                statement_qualifiers = qualifiers_by_statement_and_pid.get(
                    constraint_statement,
                    {}
                )

                for qualifier_pid, qualifier_records in statement_qualifiers.items():
                    statement_data[qualifier_pid] = qualifier_records

                raw_dict[constrained_property]["statements"][constraint_statement] = (
                    statement_data
                )

        report = {
            "constraint_kind": constraint_kind,
            "constraint_type_qid": constraint_type_qid,
            "n_constraint_type_rows": len(constraint_type_rows),
            "n_constraint_statements": len(constraint_statement_ids),
            "n_property_links": len(property_links),
            "n_linked_constraint_statements": property_links["constraint_statement"].nunique(),
            "n_properties": len(raw_dict),
            "n_rank_rows": len(ranks),
            "n_ranked_constraint_statements": ranks["constraint_statement"].nunique(),
            "n_qualifier_rows": len(qualifiers),
            "qualifier_pid_counts": qualifiers["qualifier_pid"].value_counts().to_dict(),
            "n_statements_without_property_link": len(
                constraint_statement_ids - set(property_links["constraint_statement"])
            ),
            "n_statements_without_rank": len(
                constraint_statement_ids - set(ranks["constraint_statement"])
            ),
        }

        return raw_dict, report

## Build raw one-of constraint dictionary

In [ ]:
builder = HistoricalConstraintIndexBuilder(df_constraints_clean)

one_of_raw_dict, one_of_report = builder.build_raw_constraint_dict(
    constraint_kind="one_of",
    constraint_type_qid=CONSTRAINT_TYPES["one_of"]
)

one_of_report

## Inspect one example

In [ ]:
example_property = next(iter(one_of_raw_dict))
example_property, one_of_raw_dict[example_property]

In [ ]:
df_constraints_clean[ df_constraints_clean['s'] == 'wds:P10339-5dca253a-4e47-c053-ee76-a630a1cbda61' ]

## Build raw none-of constraint dictionary

In [ ]:
none_of_raw_dict, none_of_report = builder.build_raw_constraint_dict(
    constraint_kind="none_of",
    constraint_type_qid=CONSTRAINT_TYPES["none_of"]
)

none_of_report

In [ ]:
example_property_n = next(iter(none_of_raw_dict))
example_property_n, none_of_raw_dict[example_property_n]

## Save raw JSON exports

In [ ]:
with open("one_of_raw_constraints.json", "w", encoding="utf-8") as f:
    json.dump(one_of_raw_dict, f, ensure_ascii=False, indent=2)

with open("none_of_raw_constraints.json", "w", encoding="utf-8") as f:
    json.dump(none_of_raw_dict, f, ensure_ascii=False, indent=2)

## Export simplified historical constraints

We now transform the raw constraint JSON into a simplified form ready for validation against historical `wdt:` triples.

The simplified export keeps only the qualifier values that were valid when the corresponding constraint statement was active.

Rank rules:

- `wikibase:DeprecatedRank` makes the constraint inactive during that interval.
- `wikibase:NormalRank` makes the constraint active unless a preferred-rank constraint is active for the same property at the same time.
- `wikibase:PreferredRank` makes that constraint active and suppresses normal-rank constraints for the same property during the overlap.
- Statements without rank rows are reported and excluded.

The output keeps the constraint instance ID (`wds:`) for traceability.

In [20]:
import json
import pandas as pd
from collections import defaultdict

## Interval helper functions

In [21]:
NORMAL_RANK = "wikibase:NormalRank"
PREFERRED_RANK = "wikibase:PreferredRank"
DEPRECATED_RANK = "wikibase:DeprecatedRank"


def parse_time(x):
    if x is None:
        return None

    if pd.isna(x):
        return None

    if isinstance(x, str):
        x = x.strip()
        if x in {"", "NaT", "nan", "None", "null"}:
            return None

    ts = pd.to_datetime(x, errors="coerce", utc=True)

    if pd.isna(ts):
        return None

    return ts


def time_to_json(x):
    if x is None or pd.isna(x):
        return None
    return x.isoformat()


def interval_intersection(a_start, a_end, b_start, b_end):
    a_start = parse_time(a_start)
    a_end = parse_time(a_end)
    b_start = parse_time(b_start)
    b_end = parse_time(b_end)

    if a_start is None or b_start is None:
        return None

    start = max(a_start, b_start)

    if a_end is None and b_end is None:
        end = None
    elif a_end is None:
        end = b_end
    elif b_end is None:
        end = a_end
    else:
        end = min(a_end, b_end)

    if end is not None and start >= end:
        return None

    return start, end


def merge_intervals(intervals):
    intervals = [
        (parse_time(start), parse_time(end))
        for start, end in intervals
        if parse_time(start) is not None
    ]

    if not intervals:
        return []

    max_ts = pd.Timestamp.max.tz_localize("UTC")

    intervals = sorted(
        intervals,
        key=lambda x: (x[0], max_ts if x[1] is None else x[1])
    )

    merged = [intervals[0]]

    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]

        last_end_cmp = max_ts if last_end is None else last_end

        if start <= last_end_cmp:
            if last_end is None or end is None:
                merged[-1] = (last_start, None)
            else:
                merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    return merged


def subtract_interval(base_interval, sub_interval):
    base_start, base_end = base_interval
    sub_start, sub_end = sub_interval

    inter = interval_intersection(base_start, base_end, sub_start, sub_end)

    if inter is None:
        return [base_interval]

    inter_start, inter_end = inter

    pieces = []

    if base_start < inter_start:
        pieces.append((base_start, inter_start))

    if inter_end is not None:
        if base_end is None or inter_end < base_end:
            pieces.append((inter_end, base_end))

    return pieces


def subtract_intervals(base_intervals, intervals_to_subtract):
    remaining = base_intervals[:]

    for sub in intervals_to_subtract:
        new_remaining = []

        for base in remaining:
            new_remaining.extend(subtract_interval(base, sub))

        remaining = new_remaining

        if not remaining:
            break

    return merge_intervals(remaining)

## Compute active intervals for constraint statements

For each constraint statement, we first compute the structural interval:

constraint type interval ∩ property link interval

In [22]:
def get_base_intervals(statement_data):
    """
    Compute structural validity intervals for a constraint statement:

        constraint_type_intervals ∩ property_link_intervals

    Older raw files may contain a single property_link_cdate/property_link_ddate.
    New raw files contain property_link_intervals.
    """
    intervals = []

    property_link_intervals = statement_data.get("property_link_intervals")

    # Backward compatibility with older raw exports
    if property_link_intervals is None:
        property_link_intervals = [
            {
                "property_link_cdate": statement_data.get("property_link_cdate"),
                "property_link_ddate": statement_data.get("property_link_ddate"),
            }
        ]

    for type_interval in statement_data.get("constraint_type_intervals", []):
        for property_link in property_link_intervals:
            inter = interval_intersection(
                type_interval.get("constraint_cdate"),
                type_interval.get("constraint_ddate"),
                property_link.get("property_link_cdate"),
                property_link.get("property_link_ddate"),
            )

            if inter is not None:
                intervals.append(inter)

    return merge_intervals(intervals)


def get_rank_intervals(statement_data):
    rank_groups = {
        NORMAL_RANK: [],
        PREFERRED_RANK: [],
        DEPRECATED_RANK: [],
    }

    for rank_row in statement_data.get("wikibase:rank", []):
        rank = rank_row.get("rank")
        start = parse_time(rank_row.get("rank_cdate"))
        end = parse_time(rank_row.get("rank_ddate"))

        if start is None:
            continue

        rank_groups.setdefault(rank, []).append((start, end))

    for rank in rank_groups:
        rank_groups[rank] = merge_intervals(rank_groups[rank])

    return rank_groups


def get_statement_ranked_intervals(statement_data):
    base_intervals = get_base_intervals(statement_data)
    rank_intervals = get_rank_intervals(statement_data)

    normal_intervals = []
    preferred_intervals = []

    for base_start, base_end in base_intervals:
        for rank_start, rank_end in rank_intervals.get(NORMAL_RANK, []):
            inter = interval_intersection(base_start, base_end, rank_start, rank_end)
            if inter is not None:
                normal_intervals.append(inter)

        for rank_start, rank_end in rank_intervals.get(PREFERRED_RANK, []):
            inter = interval_intersection(base_start, base_end, rank_start, rank_end)
            if inter is not None:
                preferred_intervals.append(inter)

    return {
        "normal": merge_intervals(normal_intervals),
        "preferred": merge_intervals(preferred_intervals),
    }


def resolve_active_intervals_for_property(statements):
    ranked_by_statement = {}
    all_preferred_intervals = []

    for constraint_id, statement_data in statements.items():
        ranked = get_statement_ranked_intervals(statement_data)
        ranked_by_statement[constraint_id] = ranked
        all_preferred_intervals.extend(ranked["preferred"])

    all_preferred_intervals = merge_intervals(all_preferred_intervals)

    active_by_statement = {}

    for constraint_id, ranked in ranked_by_statement.items():
        preferred = ranked["preferred"]

        normal = subtract_intervals(
            ranked["normal"],
            all_preferred_intervals
        )

        active = merge_intervals(preferred + normal)

        if active:
            active_by_statement[constraint_id] = active

    return active_by_statement

## Simplified export function

This function exports one selected qualifier PID, such as:

- `pq:P2305` for one-of and none-of values
- `pq:P2303` for exceptions, if needed later

Each exported value includes:

- entity
- start date
- end date
- constraint ID

In [23]:
def build_simplified_export_from_raw(
    raw_dict,
    qualifier_pid="pq:P2305",
):
    simplified = {}

    report = {
        "properties_total": len(raw_dict),
        "properties_exported": 0,
        "statements_total": 0,
        "statements_without_rank": [],
        "statements_with_preferred_rank": [],
        "properties_with_preferred_rank": [],
        "statements_with_only_deprecated_rank": [],
    }

    for prop, prop_data in raw_dict.items():
        statements = prop_data.get("statements", {})
        report["statements_total"] += len(statements)

        prop_has_preferred = False

        for constraint_id, statement_data in statements.items():
            ranks = statement_data.get("wikibase:rank", [])

            if not ranks:
                report["statements_without_rank"].append({
                    "property": prop,
                    "constraint_id": constraint_id,
                })
                continue

            rank_values = {r.get("rank") for r in ranks}

            if PREFERRED_RANK in rank_values:
                prop_has_preferred = True
                report["statements_with_preferred_rank"].append({
                    "property": prop,
                    "constraint_id": constraint_id,
                })

        if prop_has_preferred:
            report["properties_with_preferred_rank"].append(prop)

        active_by_statement = resolve_active_intervals_for_property(statements)

        if not active_by_statement:
            for constraint_id, statement_data in statements.items():
                ranks = statement_data.get("wikibase:rank", [])
                rank_values = {r.get("rank") for r in ranks}

                if ranks and rank_values == {DEPRECATED_RANK}:
                    report["statements_with_only_deprecated_rank"].append({
                        "property": prop,
                        "constraint_id": constraint_id,
                    })

            continue

        exported_rows = []

        for constraint_id, active_intervals in active_by_statement.items():
            statement_data = statements[constraint_id]

            for qualifier_row in statement_data.get(qualifier_pid, []):
                for active_start, active_end in active_intervals:
                    inter = interval_intersection(
                        active_start,
                        active_end,
                        qualifier_row.get("value_cdate"),
                        qualifier_row.get("value_ddate"),
                    )

                    if inter is None:
                        continue

                    exported_rows.append({
                        "entity": qualifier_row.get("entity"),
                        "start_date": time_to_json(inter[0]),
                        "end_date": time_to_json(inter[1]),
                        "constraint_id": constraint_id,
                    })

        if exported_rows:
            exported_rows = sorted(
                exported_rows,
                key=lambda x: (
                    x["entity"] or "",
                    x["start_date"] or "",
                    x["constraint_id"] or "",
                )
            )

            simplified[prop] = {
                "constraint_kind": prop_data.get("constraint_kind"),
                qualifier_pid: exported_rows,
            }

    report["properties_exported"] = len(simplified)

    return simplified, report

## Build simplified one-of export

For one-of constraints, `pq:P2305` stores the allowed values.

In [ ]:
one_of_simplified, one_of_simplified_report = build_simplified_export_from_raw(
    one_of_raw_dict,
    qualifier_pid="pq:P2305",
)

one_of_simplified_report

## Build simplified none-of export

For none-of constraints, `pq:P2305` stores the forbidden values.

In [ ]:
none_of_simplified, none_of_simplified_report = build_simplified_export_from_raw(
    none_of_raw_dict,
    qualifier_pid="pq:P2305",
)

none_of_simplified_report

## Inspect preferred-rank cases

In [ ]:
{
    "one_of_properties_with_preferred_rank": len(one_of_simplified_report["properties_with_preferred_rank"]),
    "one_of_statements_with_preferred_rank": len(one_of_simplified_report["statements_with_preferred_rank"]),
    "none_of_properties_with_preferred_rank": len(none_of_simplified_report["properties_with_preferred_rank"]),
    "none_of_statements_with_preferred_rank": len(none_of_simplified_report["statements_with_preferred_rank"]),
}

## Save simplified exports

In [ ]:
with open("one_of_simplified_constraints.json", "w", encoding="utf-8") as f:
    json.dump(one_of_simplified, f, ensure_ascii=False, indent=2)

with open("none_of_simplified_constraints.json", "w", encoding="utf-8") as f:
    json.dump(none_of_simplified, f, ensure_ascii=False, indent=2)

## generating ALL raw files for the target constraints

In [ ]:
builder = HistoricalConstraintIndexBuilder(df_constraints_clean)

for kind, qid in CONSTRAINT_TYPES.items():
    print(f"Building raw for {kind} ({qid})")

    raw_dict, report = builder.build_raw_constraint_dict(
        constraint_kind=kind,
        constraint_type_qid=qid
    )

    with open(f"{kind}_raw_constraints.json", "w", encoding="utf-8") as f:
        json.dump(raw_dict, f, ensure_ascii=False, indent=2)

    with open(f"{kind}_raw_constraints_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

print("Done.")

# Build simplified constraint exports

The raw exports preserve all historical rows.  
The simplified exports keep only the intervals during which a constraint qualifier should actually be tested.

All simplified exports respect:

- constraint statement validity
- property link validity
- rank validity
- preferred-rank override
- deprecated-rank exclusion

We keep Wikidata qualifier IDs in the output.

In [24]:
import json
import pandas as pd
from collections import defaultdict

## Interval helpers

In [25]:
NORMAL_RANK = "wikibase:NormalRank"
PREFERRED_RANK = "wikibase:PreferredRank"
DEPRECATED_RANK = "wikibase:DeprecatedRank"


def parse_time(x):
    if x is None:
        return None
    if pd.isna(x):
        return None
    if isinstance(x, str):
        x = x.strip()
        if x in {"", "NaT", "nan", "None", "null"}:
            return None
    ts = pd.to_datetime(x, errors="coerce", utc=True)
    if pd.isna(ts):
        return None
    return ts


def time_to_json(x):
    if x is None or pd.isna(x):
        return None
    return x.isoformat()


def interval_intersection(a_start, a_end, b_start, b_end):
    a_start = parse_time(a_start)
    a_end = parse_time(a_end)
    b_start = parse_time(b_start)
    b_end = parse_time(b_end)

    if a_start is None or b_start is None:
        return None

    start = max(a_start, b_start)

    if a_end is None and b_end is None:
        end = None
    elif a_end is None:
        end = b_end
    elif b_end is None:
        end = a_end
    else:
        end = min(a_end, b_end)

    if end is not None and start >= end:
        return None

    return start, end


def merge_intervals(intervals):
    intervals = [
        (parse_time(start), parse_time(end))
        for start, end in intervals
        if parse_time(start) is not None
    ]

    if not intervals:
        return []

    max_ts = pd.Timestamp.max.tz_localize("UTC")
    intervals = sorted(
        intervals,
        key=lambda x: (x[0], max_ts if x[1] is None else x[1])
    )

    merged = [intervals[0]]

    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        last_end_cmp = max_ts if last_end is None else last_end

        if start <= last_end_cmp:
            if last_end is None or end is None:
                merged[-1] = (last_start, None)
            else:
                merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    return merged


def subtract_interval(base_interval, sub_interval):
    base_start, base_end = base_interval
    sub_start, sub_end = sub_interval

    inter = interval_intersection(base_start, base_end, sub_start, sub_end)

    if inter is None:
        return [base_interval]

    inter_start, inter_end = inter
    pieces = []

    if base_start < inter_start:
        pieces.append((base_start, inter_start))

    if inter_end is not None:
        if base_end is None or inter_end < base_end:
            pieces.append((inter_end, base_end))

    return pieces


def subtract_intervals(base_intervals, intervals_to_subtract):
    remaining = base_intervals[:]

    for sub in intervals_to_subtract:
        new_remaining = []
        for base in remaining:
            new_remaining.extend(subtract_interval(base, sub))
        remaining = new_remaining

        if not remaining:
            break

    return merge_intervals(remaining)

## Rank-aware active intervals

This block computes when each constraint statement is active.

Preferred-rank statements override normal-rank statements for the same property during overlapping periods.

In [26]:
def get_base_intervals(statement_data):
    """
    Compute structural validity intervals for a constraint statement:

        constraint_type_intervals ∩ property_link_intervals

    Older raw files may contain a single property_link_cdate/property_link_ddate.
    New raw files contain property_link_intervals.
    """
    intervals = []

    property_link_intervals = statement_data.get("property_link_intervals")

    # Backward compatibility with older raw exports
    if property_link_intervals is None:
        property_link_intervals = [
            {
                "property_link_cdate": statement_data.get("property_link_cdate"),
                "property_link_ddate": statement_data.get("property_link_ddate"),
            }
        ]

    for type_interval in statement_data.get("constraint_type_intervals", []):
        for property_link in property_link_intervals:
            inter = interval_intersection(
                type_interval.get("constraint_cdate"),
                type_interval.get("constraint_ddate"),
                property_link.get("property_link_cdate"),
                property_link.get("property_link_ddate"),
            )

            if inter is not None:
                intervals.append(inter)

    return merge_intervals(intervals)


def get_rank_intervals(statement_data):
    rank_groups = {
        NORMAL_RANK: [],
        PREFERRED_RANK: [],
        DEPRECATED_RANK: [],
    }

    for rank_row in statement_data.get("wikibase:rank", []):
        rank = rank_row.get("rank")
        start = parse_time(rank_row.get("rank_cdate"))
        end = parse_time(rank_row.get("rank_ddate"))

        if start is None:
            continue

        rank_groups.setdefault(rank, []).append((start, end))

    for rank in rank_groups:
        rank_groups[rank] = merge_intervals(rank_groups[rank])

    return rank_groups


def get_statement_ranked_intervals(statement_data):
    base_intervals = get_base_intervals(statement_data)
    rank_intervals = get_rank_intervals(statement_data)

    normal_intervals = []
    preferred_intervals = []

    for base_start, base_end in base_intervals:
        for rank_start, rank_end in rank_intervals.get(NORMAL_RANK, []):
            inter = interval_intersection(base_start, base_end, rank_start, rank_end)
            if inter is not None:
                normal_intervals.append(inter)

        for rank_start, rank_end in rank_intervals.get(PREFERRED_RANK, []):
            inter = interval_intersection(base_start, base_end, rank_start, rank_end)
            if inter is not None:
                preferred_intervals.append(inter)

    return {
        "normal": merge_intervals(normal_intervals),
        "preferred": merge_intervals(preferred_intervals),
    }


def resolve_active_intervals_for_property(statements):
    ranked_by_statement = {}
    all_preferred_intervals = []

    for constraint_id, statement_data in statements.items():
        ranked = get_statement_ranked_intervals(statement_data)
        ranked_by_statement[constraint_id] = ranked
        all_preferred_intervals.extend(ranked["preferred"])

    all_preferred_intervals = merge_intervals(all_preferred_intervals)

    active_by_statement = {}

    for constraint_id, ranked in ranked_by_statement.items():
        preferred = ranked["preferred"]
        normal = subtract_intervals(ranked["normal"], all_preferred_intervals)
        active = merge_intervals(preferred + normal)

        if active:
            active_by_statement[constraint_id] = active

    return active_by_statement

## Shared export helpers

In [27]:
def build_rank_report(raw_dict):
    report = {
        "properties_total": len(raw_dict),
        "properties_exported": 0,
        "statements_total": 0,
        "statements_without_rank": [],
        "statements_with_preferred_rank": [],
        "properties_with_preferred_rank": [],
        "statements_with_only_deprecated_rank": [],
    }

    for prop, prop_data in raw_dict.items():
        statements = prop_data.get("statements", {})
        report["statements_total"] += len(statements)

        prop_has_preferred = False

        for constraint_id, statement_data in statements.items():
            ranks = statement_data.get("wikibase:rank", [])

            if not ranks:
                report["statements_without_rank"].append({
                    "property": prop,
                    "constraint_id": constraint_id,
                })
                continue

            rank_values = {r.get("rank") for r in ranks}

            if PREFERRED_RANK in rank_values:
                prop_has_preferred = True
                report["statements_with_preferred_rank"].append({
                    "property": prop,
                    "constraint_id": constraint_id,
                })

            if rank_values == {DEPRECATED_RANK}:
                report["statements_with_only_deprecated_rank"].append({
                    "property": prop,
                    "constraint_id": constraint_id,
                })

        if prop_has_preferred:
            report["properties_with_preferred_rank"].append(prop)

    return report


def export_qualifier_intervals(statement_data, constraint_id, active_intervals, qualifier_pid):
    rows = []

    for qualifier_row in statement_data.get(qualifier_pid, []):
        for active_start, active_end in active_intervals:
            inter = interval_intersection(
                active_start,
                active_end,
                qualifier_row.get("value_cdate"),
                qualifier_row.get("value_ddate"),
            )

            if inter is None:
                continue

            rows.append({
                "entity": qualifier_row.get("entity"),
                "start_date": time_to_json(inter[0]),
                "end_date": time_to_json(inter[1]),
                "constraint_id": constraint_id,
            })

    return rows


def sort_export_rows(rows, keys=("entity", "start_date", "constraint_id")):
    return sorted(
        rows,
        key=lambda x: tuple(x.get(k) or "" for k in keys)
    )

## Simplified export: value-list constraints

Used for:

- one-of
- none-of

The relevant qualifier is `pq:P2305`.

In [28]:
def build_value_list_simplified_export(raw_dict, qualifier_pid="pq:P2305"):
    simplified = {}
    report = build_rank_report(raw_dict)

    for prop, prop_data in raw_dict.items():
        statements = prop_data.get("statements", {})
        active_by_statement = resolve_active_intervals_for_property(statements)

        value_rows = []
        exception_rows = []

        for constraint_id, active_intervals in active_by_statement.items():
            statement_data = statements[constraint_id]

            value_rows.extend(
                export_qualifier_intervals(
                    statement_data,
                    constraint_id,
                    active_intervals,
                    qualifier_pid,
                )
            )

            exception_rows.extend(
                export_qualifier_intervals(
                    statement_data,
                    constraint_id,
                    active_intervals,
                    "pq:P2303",
                )
            )

        if value_rows or exception_rows:
            simplified[prop] = {
                "constraint_kind": prop_data.get("constraint_kind"),
                qualifier_pid: sort_export_rows(value_rows),
                "pq:P2303": sort_export_rows(exception_rows),
            }

    report["properties_exported"] = len(simplified)
    return simplified, report

## Simplified export: symmetric constraint

Symmetric constraints need only the active constraint intervals and exceptions.

The relevant exception qualifier is `pq:P2303`.

In [29]:
def build_symmetric_simplified_export(raw_dict):
    simplified = {}
    report = build_rank_report(raw_dict)

    for prop, prop_data in raw_dict.items():
        statements = prop_data.get("statements", {})
        active_by_statement = resolve_active_intervals_for_property(statements)

        active_rows = []
        exception_rows = []

        for constraint_id, active_intervals in active_by_statement.items():
            statement_data = statements[constraint_id]

            for start, end in active_intervals:
                active_rows.append({
                    "start_date": time_to_json(start),
                    "end_date": time_to_json(end),
                    "constraint_id": constraint_id,
                })

            exception_rows.extend(
                export_qualifier_intervals(
                    statement_data,
                    constraint_id,
                    active_intervals,
                    "pq:P2303",
                )
            )

        if active_rows:
            simplified[prop] = {
                "constraint_kind": prop_data.get("constraint_kind"),
                "active_intervals": sort_export_rows(
                    active_rows,
                    keys=("start_date", "constraint_id"),
                ),
                "pq:P2303": sort_export_rows(exception_rows),
            }

    report["properties_exported"] = len(simplified)
    return simplified, report

## Simplified export: inverse constraint

Inverse constraints use:

- `pq:P2306` for the inverse property
- `pq:P2303` for exceptions

In [30]:
def build_inverse_simplified_export(raw_dict):
    simplified = {}
    report = build_rank_report(raw_dict)

    for prop, prop_data in raw_dict.items():
        statements = prop_data.get("statements", {})
        active_by_statement = resolve_active_intervals_for_property(statements)

        p2306_rows = []
        exception_rows = []

        for constraint_id, active_intervals in active_by_statement.items():
            statement_data = statements[constraint_id]

            p2306_rows.extend(
                export_qualifier_intervals(
                    statement_data,
                    constraint_id,
                    active_intervals,
                    "pq:P2306",
                )
            )

            exception_rows.extend(
                export_qualifier_intervals(
                    statement_data,
                    constraint_id,
                    active_intervals,
                    "pq:P2303",
                )
            )

        if p2306_rows:
            simplified[prop] = {
                "constraint_kind": prop_data.get("constraint_kind"),
                "pq:P2306": sort_export_rows(p2306_rows),
                "pq:P2303": sort_export_rows(exception_rows),
            }

    report["properties_exported"] = len(simplified)
    return simplified, report

## Pair-based helper for P2306 and P2305

Used for:

- item-requires-statement
- value-requires-statement
- conflicts-with

Each row keeps Wikidata qualifier IDs:

- `pq:P2306`
- `pq:P2305`

If a required/conflicting property exists without a simultaneous `pq:P2305`, the exported row has:

`
"pq:P2305": None
`

In [31]:
def build_p2306_p2305_pair_rows(statement_data, constraint_id, active_intervals):
    rows = []

    p2306_rows = statement_data.get("pq:P2306", [])
    p2305_rows = statement_data.get("pq:P2305", [])

    for p2306 in p2306_rows:
        for active_start, active_end in active_intervals:
            p2306_active = interval_intersection(
                active_start,
                active_end,
                p2306.get("value_cdate"),
                p2306.get("value_ddate"),
            )

            if p2306_active is None:
                continue

            p2306_start, p2306_end = p2306_active

            overlapping_value_intervals = []

            for p2305 in p2305_rows:
                value_inter = interval_intersection(
                    p2306_start,
                    p2306_end,
                    p2305.get("value_cdate"),
                    p2305.get("value_ddate"),
                )

                if value_inter is not None:
                    overlapping_value_intervals.append(value_inter)

                    rows.append({
                        "pq:P2306": p2306.get("entity"),
                        "pq:P2305": p2305.get("entity"),
                        "start_date": time_to_json(value_inter[0]),
                        "end_date": time_to_json(value_inter[1]),
                        "constraint_id": constraint_id,
                    })

            property_only_intervals = subtract_intervals(
                [p2306_active],
                overlapping_value_intervals,
            )

            for start, end in property_only_intervals:
                rows.append({
                    "pq:P2306": p2306.get("entity"),
                    "pq:P2305": None,
                    "start_date": time_to_json(start),
                    "end_date": time_to_json(end),
                    "constraint_id": constraint_id,
                })

    return rows


def build_p2306_p2305_simplified_export(raw_dict, output_key="requirements"):
    simplified = {}
    report = build_rank_report(raw_dict)

    for prop, prop_data in raw_dict.items():
        statements = prop_data.get("statements", {})
        active_by_statement = resolve_active_intervals_for_property(statements)

        pair_rows = []
        exception_rows = []

        for constraint_id, active_intervals in active_by_statement.items():
            statement_data = statements[constraint_id]

            pair_rows.extend(
                build_p2306_p2305_pair_rows(
                    statement_data,
                    constraint_id,
                    active_intervals,
                )
            )

            exception_rows.extend(
                export_qualifier_intervals(
                    statement_data,
                    constraint_id,
                    active_intervals,
                    "pq:P2303",
                )
            )

        if pair_rows:
            simplified[prop] = {
                "constraint_kind": prop_data.get("constraint_kind"),
                output_key: sorted(
                    pair_rows,
                    key=lambda x: (
                        x.get("pq:P2306") or "",
                        x.get("pq:P2305") or "",
                        x.get("start_date") or "",
                        x.get("constraint_id") or "",
                    )
                ),
                "pq:P2303": sort_export_rows(exception_rows),
            }

    report["properties_exported"] = len(simplified)
    return simplified, report

## Dispatcher for simplified exports

In [32]:
def build_simplified_export(raw_dict, constraint_kind):
    if constraint_kind in {"one_of", "none_of"}:
        return build_value_list_simplified_export(
            raw_dict,
            qualifier_pid="pq:P2305",
        )

    if constraint_kind == "symmetric":
        return build_symmetric_simplified_export(raw_dict)

    if constraint_kind == "inverse":
        return build_inverse_simplified_export(raw_dict)

    if constraint_kind in {
        "item_requires_statement",
        "value_requires_statement",
    }:
        return build_p2306_p2305_simplified_export(
            raw_dict,
            output_key="requirements",
        )

    if constraint_kind == "conflicts_with":
        return build_p2306_p2305_simplified_export(
            raw_dict,
            output_key="conflicts",
        )

    raise ValueError(f"Unsupported constraint kind: {constraint_kind}")

## Build all

In [34]:
builder = HistoricalConstraintIndexBuilder(df_constraints_clean)

for kind, qid in CONSTRAINT_TYPES.items():
    print(f"Building raw for {kind} ({qid})")

    raw_dict, report = builder.build_raw_constraint_dict(
        constraint_kind=kind,
        constraint_type_qid=qid
    )

    with open(f"constraint_files/{kind}_raw_constraints.json", "w", encoding="utf-8") as f:
        json.dump(raw_dict, f, ensure_ascii=False, indent=2)

print("Raw done.")

Building raw for one_of (wd:Q21510859)
Building raw for none_of (wd:Q52558054)
Building raw for symmetric (wd:Q21510862)
Building raw for inverse (wd:Q21510855)
Building raw for item_requires_statement (wd:Q21503247)
Building raw for value_requires_statement (wd:Q21510864)
Building raw for conflicts_with (wd:Q21502838)
Raw done.


In [35]:
simplified_reports = {}

for kind in CONSTRAINT_TYPES:
    raw_path = f"constraint_files/{kind}_raw_constraints.json"

    print(f"Building simplified export for {kind}")

    with open(raw_path, "r", encoding="utf-8") as f:
        raw_dict = json.load(f)

    simplified, report = build_simplified_export(
        raw_dict,
        constraint_kind=kind,
    )

    with open(f"constraint_files/{kind}_simplified_constraints.json", "w", encoding="utf-8") as f:
        json.dump(simplified, f, ensure_ascii=False, indent=2)

    simplified_reports[kind] = report

print("Simplified done.")

Building simplified export for one_of
Building simplified export for none_of
Building simplified export for symmetric
Building simplified export for inverse
Building simplified export for item_requires_statement
Building simplified export for value_requires_statement
Building simplified export for conflicts_with
Simplified done.


## Tests

In [36]:
df_constraints_clean[(df_constraints_clean['s'] == "wd:P460") & (df_constraints_clean['o'] == "wds:P460-FB1E15A6-C38C-4AF7-A6D4-346739D2AA32")]


,s,p,o,cdate,cuser,ddate,duser
464269,wd:P460,p:P2302,wds:P460-FB1E15A6-C38C-4AF7-A6D4-346739D2AA32,2017-07-13 04:30:14+00:00,KrBot,2019-01-29 02:52:04+00:00,123.136.111.27
464280,wd:P460,p:P2302,wds:P460-FB1E15A6-C38C-4AF7-A6D4-346739D2AA32,2019-01-29 06:16:53+00:00,Ayack,2019-06-15 19:47:02+00:00,135.19.173.112
464283,wd:P460,p:P2302,wds:P460-FB1E15A6-C38C-4AF7-A6D4-346739D2AA32,2019-06-15 20:45:30+00:00,Laddo,2020-10-11 13:57:35+00:00,5.197.238.209
464329,wd:P460,p:P2302,wds:P460-FB1E15A6-C38C-4AF7-A6D4-346739D2AA32,2020-10-11 17:29:29+00:00,Shinnin,NaT,NaN


In [37]:
df_constraints_clean[(df_constraints_clean['s'] == "wd:P21") & (df_constraints_clean['o'] == "wds:P21-09D3E4D3-BBC5-4F40-9BB7-CC96C7721A56")]


,s,p,o,cdate,cuser,ddate,duser
412928,wd:P21,p:P2302,wds:P21-09D3E4D3-BBC5-4F40-9BB7-CC96C7721A56,2017-07-13 07:22:51+00:00,KrBot,NaT,NaN


In [39]:
df_constraints_clean[(df_constraints_clean['s'] == "wd:P1196") & (df_constraints_clean['o'] == "wds:P1196-0B380D8D-FCE0-4DDF-BB14-F40DC95A0974")]


,s,p,o,cdate,cuser,ddate,duser
41173,wd:P1196,p:P2302,wds:P1196-0B380D8D-FCE0-4DDF-BB14-F40DC95A0974,2017-07-13 01:29:22+00:00,KrBot,2022-12-03 07:33:27+00:00,Павло Сарт
